<a href="https://colab.research.google.com/github/jiffy-jacob/Resume-based-Interview-Preparation-application/blob/main/interview_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RESUME BASED INTERVIEW PREPARATION APP

**DEVELOPMENT PIPELINE**

             1. Install libraries
                      ↓
             2. Load Resume PDFs
                      ↓
             3. Extract text
                      ↓
             4. Clean text
                      ↓
             5. Chunk documents
                      ↓
             6. Generate MiniLM embeddings
                      ↓
             7. Create FAISS vector store
                      ↓
             8. Build Retrieval System
                      ↓
             9. Question Generation
                      ↓
            10. Answer Evaluation
                      ↓
            11. Save required files
                      │
                      ↓
              GOOGLE DRIVE / GitHub
                      │
                      ↓
                 STREAMLIT
                      │
                      ↓
              Final Interview App

**1. Install Libraries**

In [ ]:
!pip install -q requests==2.32.4

In [ ]:
import requests
print(requests.__version__)

2.32.4


**Import Libraries**

In [ ]:
!pip install -q pypdf langchain sentence-transformers faiss-cpu transformers

import pypdf
import langchain
import sentence_transformers
import faiss
import transformers

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
from pypdf import PdfReader
import os
import re

**Create Folders**

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#create project folders in google drive
import os

project_path = "/content/drive/MyDrive/AI_Interview_Assistant"

folders = [
    "data/resume",
    "data/knowledge_base",
    "preprocessing",
    "embeddings",
    "vector_store"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [ ]:
#Set project path
PROJECT_PATH = "/content/drive/MyDrive/AI_Interview_Assistant"

RESUME_PATH = os.path.join(PROJECT_PATH, "data/resume")
KNOWLEDGE_BASE_PATH = os.path.join(PROJECT_PATH, "data/knowledge_base")
VECTOR_STORE_PATH = os.path.join(PROJECT_PATH, "vector_store")

print("Project path:", PROJECT_PATH)

Project path: /content/drive/MyDrive/AI_Interview_Assistant


**2. Extract Text from uploaded resume pdf**

In [ ]:
from pypdf import PdfReader
import os

RESUME_PATH = "/content/drive/MyDrive/AI_Interview_Assistant/data/resume"

# Find the uploaded PDF
pdf_files = [f for f in os.listdir(RESUME_PATH) if f.lower().endswith(".pdf")]

if not pdf_files:
    print("No PDF found. Please check your resume folder.")
else:
    pdf_path = os.path.join(RESUME_PATH, pdf_files[0])

    reader = PdfReader(pdf_path)

    resume_text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            resume_text += page_text + "\n"

    print("Resume extracted successfully!")
    print("Number of pages:", len(reader.pages))
    print("\nExtracted Resume Text:\n")
    print(resume_text)

Resume extracted successfully!
Number of pages: 1

Extracted Resume Text:

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 +91-8547765901 | Kannur, Kerala | jiffy.jacob1410@gmail.com
 linkedin.com/in/jiffy-jacob | github.com/jiffy-jacob
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior

Here,
- PdfReader opens our resume PDF.
- extract_text() retrieves the text from each page.
- resume_text combines the extracted text into one string.
- Finally, we display the extracted text so we can check its quality.

**3. Clean the Extracted Text**

In [ ]:
import re

# Create a copy of the extracted text
cleaned_resume_text = resume_text

# Replace unusual bullet characters with standard bullets
cleaned_resume_text = cleaned_resume_text.replace("", "•")

# Remove extra spaces within lines
cleaned_resume_text = re.sub(r"[ \t]+", " ", cleaned_resume_text)

# Remove unnecessary blank lines
cleaned_resume_text = re.sub(r"\n\s*\n+", "\n\n", cleaned_resume_text)

# Remove leading and trailing whitespace
cleaned_resume_text = cleaned_resume_text.strip()

print("Resume text cleaned successfully!")
print("\nCleaned Resume Text:\n")
print(cleaned_resume_text)

Resume text cleaned successfully!

Cleaned Resume Text:

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 +91-8547765901 | Kannur, Kerala | jiffy.jacob1410@gmail.com
 linkedin.com/in/jiffy-jacob | github.com/jiffy-jacob
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 • Buil

During cleaning, we
- Replace unusual bullet characters	to make bullet points readable
- Remove extra spaces	to improve text consistency
- Remove unnecessary blank lines	to make the extracted text more organized
-Strip whitespace	to remove spaces at the beginning and end.
- We are not removing stop words, numbers, or technical terms. Words such as Python, Random Forest, 92.85%, and Hit Rate@10 are important for generating personalized interview questions.

In [ ]:
#Save cleaned Text
cleaned_file_path = os.path.join(
    PROJECT_PATH,
    "preprocessing",
    "cleaned_resume.txt"
)

with open(cleaned_file_path, "w", encoding="utf-8") as file:
    file.write(cleaned_resume_text)

print("Cleaned resume saved to:", cleaned_file_path)

Cleaned resume saved to: /content/drive/MyDrive/AI_Interview_Assistant/preprocessing/cleaned_resume.txt


**4. Divide the resume into meaningful sections**

*This will help our system identify which information relates to skills, projects, education, and other areas.*

In [ ]:
import re

# Define the resume section headings
section_headings = [
    "PROFESSIONAL SUMMARY",
    "TECHNICAL SKILLS",
    "MACHINE LEARNING PROJECTS",
    "CERTIFICATION",
    "EDUCATION"
]

# Split the resume using the section headings
pattern = r"(?=(" + "|".join(map(re.escape, section_headings)) + r"))"

parts = re.split(pattern, cleaned_resume_text)

# Store the sections in a dictionary
resume_sections = {}

for i in range(1, len(parts), 2):
    heading = parts[i].strip()
    content = parts[i + 1].strip() if i + 1 < len(parts) else ""
    resume_sections[heading] = content

# Display the extracted sections
for heading, content in resume_sections.items():
    print(f"\n{'=' * 50}")
    print(heading)
    print('=' * 50)
    print(content)


PROFESSIONAL SUMMARY
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.

TECHNICAL SKILLS
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub

MACHINE LEARNING PROJECTS
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 • Built sales prediction models on Olist e-commerce data using preprocessing, EDA, feature
 engineering, and five ML algorithms.
• Random Forest achieved 

Here,
- It identifies headings: Finds the main sections in your resume.
- It splits the text: Separates the content belonging to each section.
- It creates a dictionary: Stores each heading alongside its content.

**5. Preprare the Knowledge base pdfs**

In [ ]:
import os
from pypdf import PdfReader

KNOWLEDGE_BASE_PATH = os.path.join(
    PROJECT_PATH, "data", "knowledge_base"
)

documents = []

for filename in os.listdir(KNOWLEDGE_BASE_PATH):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(KNOWLEDGE_BASE_PATH, filename)

        reader = PdfReader(pdf_path)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

        documents.append({
            "source": filename,
            "text": text.strip()
        })

print("Total PDFs loaded:", len(documents))

for document in documents:
    print(f"\nFile: {document['source']}")
    print("Characters extracted:", len(document["text"]))

Total PDFs loaded: 5

File: hr_behavioral_interview_questions.pdf
Characters extracted: 6326

File: ai_ml_engineer_interview_guide.pdf
Characters extracted: 6668

File: resume_project_deep_dive_guide.pdf
Characters extracted: 6026

File: data_scientist_interview_guide.pdf
Characters extracted: 7159

File: ml_interview_knowledge_base.pdf
Characters extracted: 18346


In [ ]:
!pip install -q langchain-text-splitters

#Split Text into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split each document into chunks
new_chunks = []

for document in documents:
    chunks = text_splitter.split_text(document["text"])

    for chunk in chunks:
        new_chunks.append({
            "source": document["source"],
            "text": chunk
        })

print("Previous number of chunks:", len(loaded_chunks))
print("New number of chunks:", len(new_chunks))

Previous number of chunks: 111
New number of chunks: 74


**6. Generate Embeddings Using MiniLM**

*We'll use the pretrained all-MiniLM-L6-v2 model.*

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the pretrained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract the text from all chunks
chunk_texts = [chunk["text"] for chunk in new_chunks]

# Convert text into numerical vectors
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)
embeddings = np.asarray(
    embeddings,
    dtype="float32"
)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Number of chunks: 74
Embedding shape: (74, 384)


**7. Building the FAISS vector database**

In [ ]:
#Creating FAISS Index
import faiss
import numpy as np
import os
import pickle

# Convert embeddings to the required format
embeddings = np.asarray(embeddings, dtype="float32")

# Create the FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

# Add embeddings to the index
index.add(embeddings)

print("FAISS index created successfully!")
print("Number of vectors:", index.ntotal)
print("Embedding dimension:", index.d)

FAISS index created successfully!
Number of vectors: 74
Embedding dimension: 384


In [ ]:
# Save the updated index
index_path = os.path.join(
    PROJECT_PATH, "vector_store", "interview_index.faiss"
)

chunks_path = os.path.join(
    PROJECT_PATH, "vector_store", "knowledge_chunks.pkl"
)

faiss.write_index(index, index_path)

with open(chunks_path, "wb") as f:
    pickle.dump(new_chunks, f)

print("\nUpdated index and chunks saved successfully!")


Updated index and chunks saved successfully!


In [ ]:
#Reload updated Index
import faiss
import pickle

loaded_index = faiss.read_index(index_path)

with open(chunks_path, "rb") as f:
    loaded_chunks = pickle.load(f)

print("Index loaded successfully!")
print("Number of vectors:", loaded_index.ntotal)
print("Number of chunks:", len(loaded_chunks))

Index loaded successfully!
Number of vectors: 74
Number of chunks: 74


In [ ]:
#Create the retrieval function to find relevant information from the knowledge base when a user asks a question.
def retrieve_context(query, k=3):

    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search for the most relevant chunks
    distances, indices = index.search(query_embedding, k)

    retrieved_chunks = []

    for idx in indices[0]:
        if idx != -1:
            chunk = new_chunks[idx]

            retrieved_chunks.append({
                "source": chunk["source"],
                "text": chunk["text"]
            })

    return retrieved_chunks

In [ ]:
#Test RAG Retrieval
query = "How does a decision tree work, and what is a Random Forest?"

results = retrieve_context(query, k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Source:", result["source"])
    print("Text:", result["text"])


--- Retrieved Chunk 1 ---
Source: ml_interview_knowledge_base.pdf
Text: 5. How does a decision tree work, and what is a Random Forest?
Answer framework A decision tree repeatedly splits examples using feature rules to make groups more predictive of the
target. A Random Forest averages many trees trained on varied samples and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can a deep tree overfit? / How can you estimate feature importance?
6. When might you use KNN or an SVM?
Answer framework K-nearest neighbors predicts from the labels or values of nearby training examples, so feature scaling

--- Retrieved Chunk 2 ---
Source: ml_interview_knowledge_base.pdf
Text: how each feature relates to price while other features are held fixed.
Likely follow-ups What does a coefficient mean? / What assu

In [ ]:
#Test the Retrieval Function with Different Questions
test_queries = [
    "What is supervised learning?",
    "Explain overfitting and underfitting.",
    "How do you evaluate a classification model?"
]

for query in test_queries:
    print("\n" + "=" * 60)
    print("QUERY:", query)
    print("=" * 60)

    results = retrieve_context(query, k=2)

    for i, result in enumerate(results, 1):
        print(f"\nChunk {i} | Source: {result['source']}")
        print(result["text"][:500])


QUERY: What is supervised learning?

Chunk 1 | Source: ml_interview_knowledge_base.pdf
ML Interview Knowledge Base
2
1. Machine learning fundamentals and algorithms
1. How do supervised and unsupervised learning differ?
Answer framework Supervised learning uses examples with known target answers to learn a mapping from inputs to
outputs. Unsupervised learning looks for structure in inputs without target labels.
Fresher example For example, predict whether a customer will buy using past labeled purchases (supervised), or group
customers by similar behavior (unsupervised).
Likely fo

Chunk 2 | Source: ml_interview_knowledge_base.pdf
ML Interview Knowledge Base
4
6. What is feature engineering?
Answer framework Feature engineering creates or transforms inputs so that relevant patterns are easier for a model to
learn. It should use information available at prediction time and be evaluated on held-out data.
Fresher example For a purchase prediction, useful features might include days since

# Load the saved FAISS index

In [ ]:
import os
import pickle
import faiss

# Define the saved file paths
index_path = os.path.join(
    PROJECT_PATH,
    "vector_store",
    "interview_index.faiss"
)

chunks_path = os.path.join(
    PROJECT_PATH,
    "vector_store",
    "knowledge_chunks.pkl"
)

# Load the FAISS index
loaded_index = faiss.read_index(index_path)

# Load the knowledge chunks
with open(chunks_path, "rb") as f:
    loaded_chunks = pickle.load(f)

print("FAISS index loaded successfully!")
print("Number of vectors:", loaded_index.ntotal)
print("Number of knowledge chunks:", len(loaded_chunks))

FAISS index loaded successfully!
Number of vectors: 111
Number of knowledge chunks: 111


In [ ]:
#Test retrieval using the loaded index
def retrieve_context(query, k=3):

    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search the loaded FAISS index
    distances, indices = loaded_index.search(
        query_embedding, k
    )

    retrieved_chunks = []

    for idx in indices[0]:
        if idx != -1:
            chunk = loaded_chunks[idx]

            retrieved_chunks.append({
                "source": chunk["source"],
                "text": chunk["text"]
            })

    return retrieved_chunks


# Test the function
results = retrieve_context(
    "How does Random Forest work?",
    k=3
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Text:", result["text"])
    print("-" * 50)


Result 1
Source: ml_interview_knowledge_base.pdf
Text: target. A Random Forest averages many trees trained on varied samples and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can a deep tree overfit? / How can you estimate feature importance?
6. When might you use KNN or an SVM?
--------------------------------------------------

Result 2
Source: ml_interview_knowledge_base.pdf
Text: Fresher example An order-risk model may estimate a 0.78 chance of cancellation. A threshold such as 0.5 turns that into
a predicted cancellation.
Likely follow-ups How would you choose a threshold? / Why might the default threshold be unsuitable?
5. How does a decision tree work, and what is a Random Forest?
Answer framework A decision tree repeatedly splits examples using feature rules to make groups more pred

In [ ]:
# Load the cleaned resume text

resume_text_path = os.path.join(
    PROJECT_PATH,
    "preprocessing",
    "cleaned_resume.txt"
)

with open(resume_text_path, "r", encoding="utf-8") as file:
    cleaned_resume_text = file.read()

print("Resume loaded successfully!")
print("\nResume preview:\n")
print(cleaned_resume_text[:1500])

Resume loaded successfully!

Resume preview:

JIFFY JACOB
 AI / ML INTERN | DATA SCIENCE
 +91-8547765901 | Kannur, Kerala | jiffy.jacob1410@gmail.com
 linkedin.com/in/jiffy-jacob | github.com/jiffy-jacob
PROFESSIONAL SUMMARY
Data Scientist with hands-on experience in Python, data preprocessing, EDA, feature engineering,
machine learning, NLP, and model evaluation. Built predictive, recommendation, clustering, and
text-classification projects using real-world datasets.
TECHNICAL SKILLS
Programming: Python
Data & ML: Pandas, NumPy, Scikit-learn, EDA, Feature Engineering, Regression, Classification,
Clustering, Model Evaluation
Deep Learning & NLP: TensorFlow/Keras, Neural Networks, TF-IDF, NLP, Sentiment/Emotion
Classification
Recommendation: Collaborative Filtering, User-Based CF, Item-Based CF, Cosine Similarity
Tools: Matplotlib, Seaborn, Jupyter, Google Colab, Git, GitHub
MACHINE LEARNING PROJECTS
ML-Powered E-Commerce Sales Prediction & Customer Behavior Analytics
 • Built sales pre

In [ ]:
#Combine our resume with retrieved knowledge
# Select the relevant project information from your resume
project_context = """
Project: E-Commerce Sales Prediction and Customer Behavior Analytics

Dataset: Olist e-commerce dataset.

Work performed:
- Data preprocessing and exploratory data analysis.
- Feature engineering.
- Evaluation of five machine learning algorithms.
- Random Forest achieved an R2 score of approximately 0.823.
- K-Means clustering was used for customer segmentation.
"""

# Retrieve relevant knowledge from FAISS
retrieved_results = retrieve_context(
    "Random Forest algorithm, advantages, overfitting, and model evaluation",
    k=3
)

# Combine the retrieved text
retrieved_context = "\n\n".join(
    result["text"] for result in retrieved_results
)

print("PROJECT CONTEXT")
print(project_context)

print("\nRETRIEVED KNOWLEDGE")
print(retrieved_context)

PROJECT CONTEXT

Project: E-Commerce Sales Prediction and Customer Behavior Analytics

Dataset: Olist e-commerce dataset.

Work performed:
- Data preprocessing and exploratory data analysis.
- Feature engineering.
- Evaluation of five machine learning algorithms.
- Random Forest achieved an R2 score of approximately 0.823.
- K-Means clustering was used for customer segmentation.


RETRIEVED KNOWLEDGE
target. A Random Forest averages many trees trained on varied samples and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can a deep tree overfit? / How can you estimate feature importance?
6. When might you use KNN or an SVM?

Fresher example An order-risk model may estimate a 0.78 chance of cancellation. A threshold such as 0.5 turns that into
a predicted cancellation.
Likely follow-ups How woul

**8. Load Qwen2.5-1.5B-Instruct**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(model_name)

qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen model loaded successfully!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen model loaded successfully!


In [ ]:
#Test Question generation
messages = [
    {
        "role": "system",
        "content": (
            "You are an interview question generator for "
            "freshers applying for data science roles."
        )
    },
    {
        "role": "user",
        "content": (
            "Generate exactly five distinct interview questions "
            "about the Random Forest algorithm. "
            "Cover how it works, its advantages, overfitting, "
            "and its application to e-commerce sales prediction. "
            "Return only the numbered questions. "
            "Do not ask for an R2 score."
        )
    }
]

prompt_text = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = qwen_tokenizer(
    prompt_text,
    return_tensors="pt"
).to(qwen_model.device)

with torch.no_grad():
    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

generated_text = qwen_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(generated_text)

1. How does the Random Forest algorithm work in terms of decision-making compared to a single decision tree?
2. Can you explain why Random Forests are considered more robust than individual decision trees when dealing with high-dimensional datasets?
3. What is overfitting in the context of Random Forests, and how can it be mitigated?
4. In what scenarios would you apply Random Forests for predicting e-commerce sales? Provide an example.
5. How do you interpret the importance scores given by Random Forests for feature selection in predictive modeling?


In [ ]:
#Connect Qwen to our RAG retrieval function
# Step 1: Retrieve relevant knowledge-base content

query = "Random Forest interview questions for a data science fresher"

retrieved_chunks = retrieve_context(query, k=3)

print("Retrieved Knowledge-Base Content:\n")

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"Chunk {i} - Source: {chunk['source']}")
    print(chunk["text"])
    print("-" * 80)

Retrieved Knowledge-Base Content:

Chunk 1 - Source: data_scientist_interview_guide.pdf
- GeeksforGeeks - Data Science Interview Questions and Answers
https://www.geeksforgeeks.org/data-science/data-science-interview-questions-and-answers/
- GeeksforGeeks - Lowe's India Data Scientist interview experience
https://www.geeksforgeeks.org/interview-experiences/lowes-india-interview-experience-for-data-scientist-on-campus-2023/
- Medium / Towards Data Science - Notes and technical questions from interviewing as a Data Scientist
--------------------------------------------------------------------------------
Chunk 2 - Source: data_scientist_interview_guide.pdf
Resume-Based Interview Prep | Original study guide
1
Data Scientist Interview Guide
Original practice prompts and concise answer frameworks
A role-focused guide across analytics, statistics, experimentation, machine learning, and communication.
Designed for retrieval by a resume-based interview preparation app.
How to use this guide
Tr

In [ ]:
# Step 2: Generate interview questions using RAG

# Combine the retrieved chunks
context = "\n\n".join(
    chunk["text"] for chunk in retrieved_chunks
)

messages = [
    {
        "role": "system",
        "content": (
            "You are an AI interview preparation assistant "
            "for data science freshers. Generate clear, "
            "relevant interview practice questions. "
            "Use the provided context as your primary source. "
            "Do not invent facts or claim that information "
            "comes from the context when it does not."
        )
    },
    {
        "role": "user",
        "content": f"""
Generate exactly five interview questions for a
Data Science fresher.

Topic: Random Forest

Use the following retrieved knowledge-base context:

{context}

Requirements:
1. Return exactly five numbered questions.
2. Focus on Random Forest and its practical applications.
3. Do not generate answers.
4. If the retrieved context does not contain enough
   information about Random Forest, acknowledge that
   limitation instead of pretending it does.
"""
    }
]

prompt_text = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = qwen_tokenizer(
    prompt_text,
    return_tensors="pt"
).to(qwen_model.device)

with torch.no_grad():
    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

generated_text = qwen_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("RAG-Generated Interview Questions:\n")
print(generated_text)

RAG-Generated Interview Questions:

1. How do you explain the concept of Random Forests to someone who has no background in machine learning?
2. Can you describe how Random Forests handle overfitting compared to other ensemble methods like AdaBoost?
3. In what scenarios would you prefer using Random Forests over decision trees? Provide examples.
4. Explain the process of building a Random Forest model step-by-step, starting from splitting nodes until all leaves have pure classes.
5. Discuss the importance of hyperparameter tuning in Random Forest models and provide an example of how you might tune these parameters during training.


In [ ]:
# Step 3: Find Random Forest content in the knowledge base

search_terms = [
    "Random Forest",
    "decision tree",
    "overfitting",
    "feature importance",
    "hyperparameter tuning"
]

for term in search_terms:
    print(f"\n{'=' * 60}")
    print(f"Searching for: {term}")
    print("=" * 60)

    matches = []

    for i, chunk in enumerate(loaded_chunks):
        if term.lower() in chunk["text"].lower():
            matches.append((i, chunk))

    print(f"Exact text matches: {len(matches)}")

    for i, chunk in matches[:2]:
        print(f"\nChunk index: {i}")
        print(f"Source: {chunk['source']}")
        print(chunk["text"][:700])


Searching for: Random Forest
Exact text matches: 2

Chunk index: 71
Source: ml_interview_knowledge_base.pdf
Fresher example An order-risk model may estimate a 0.78 chance of cancellation. A threshold such as 0.5 turns that into
a predicted cancellation.
Likely follow-ups How would you choose a threshold? / Why might the default threshold be unsuitable?
5. How does a decision tree work, and what is a Random Forest?
Answer framework A decision tree repeatedly splits examples using feature rules to make groups more predictive of the

Chunk index: 72
Source: ml_interview_knowledge_base.pdf
target. A Random Forest averages many trees trained on varied samples and feature subsets, which usually reduces the
instability of one tree.
Fresher example A tree might first split customers by prior returns, then by order value. A forest combines many such
trees for a more stable prediction.
Likely follow-ups How can a deep tree overfit? / How can you estimate feature importance?
6. When might you us